In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter
from itertools import combinations

BASE_DIR = Path.cwd()

# Find all downloaded Parquet files
parquet_files = sorted(BASE_DIR.rglob("*.parquet"))

datasets = {
    "safeguard": [],
    "deepset": [],
    "gandalf": []
}

# Organize files by dataset
for file in parquet_files:
    folder = file.parent.name.lower()

    if "safe-guard" in folder:
        name = "safeguard"
    elif "deepset" in folder:
        name = "deepset"
    elif "gandalf" in folder:
        name = "gandalf"
    else:
        continue

    df = pd.read_parquet(file)
    df["source_file"] = file.name
    datasets[name].append(df)

# Combine partitions belonging to each dataset
for name in datasets:
    if not datasets[name]:
        raise FileNotFoundError(
            f"No Parquet files found for {name}"
        )

    datasets[name] = pd.concat(
        datasets[name],
        ignore_index=True
    )


# 1. Within-dataset duplicate audit
print("\n===== WITHIN-DATASET AUDIT =====")

counters = {}

for name, df in datasets.items():

    print(f"\nDataset: {name}")
    print("Total rows:", len(df))
    print("Columns:", df.columns.tolist())

    print("Missing text:", df["text"].isna().sum())

    if "label" in df.columns:
        print("Class counts:")
        print(df["label"].value_counts())

    counts = Counter(df["text"].dropna())
    counters[name] = counts

    duplicate_rows = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    duplicate_pairs = sum(
        count * (count - 1) // 2
        for count in counts.values()
        if count > 1
    )

    print("Duplicate extra rows:", duplicate_rows)
    print("Duplicate pairs:", duplicate_pairs)


# 2. Cross-dataset exact duplicates
print("\n===== CROSS-DATASET AUDIT =====")

for name1, name2 in combinations(datasets, 2):

    counts1 = counters[name1]
    counts2 = counters[name2]

    shared = set(counts1) & set(counts2)

    pairs = sum(
        counts1[text] * counts2[text]
        for text in shared
    )

    rows1 = sum(counts1[text] for text in shared)
    rows2 = sum(counts2[text] for text in shared)

    print(f"\n{name1} vs {name2}")
    print("Distinct shared texts:", len(shared))
    print("Cross-dataset duplicate pairs:", pairs)
    print("Rows involved in first dataset:", rows1)
    print("Rows involved in second dataset:", rows2)


#final verification of duplicates
print("\n===== SPLIT LEAKAGE CHECK =====")

for name, df in datasets.items():

    # Extract train/test/validation from filename
    temp = df.copy()
    temp["partition"] = temp["source_file"].str.split(
        "-", n=1
    ).str[0]

    # Find texts appearing in multiple partitions
    partition_counts = (
        temp.groupby("text")["partition"].nunique()
    )

    overlapping = (partition_counts > 1).sum()

    print(f"\n{name}")
    print("Texts crossing partitions:", overlapping)

    # Check identical texts with conflicting labels
    if "label" in temp.columns:
        label_counts = (
            temp.groupby("text")["label"].nunique()
        )

        conflicts = (label_counts > 1).sum()

        print("Texts with conflicting labels:", conflicts)


# 4. Check labels for cross-dataset matches
print("\n===== CROSS-DATASET LABEL CHECK =====")

safeguard = datasets["safeguard"][
    ["text", "label"]
].drop_duplicates()

deepset = datasets["deepset"][
    ["text", "label"]
].drop_duplicates()

shared = safeguard.merge(
    deepset,
    on="text",
    suffixes=("_safeguard", "_deepset")
)

print("Shared labeled records:", len(shared))

conflicts = shared[
    shared["label_safeguard"]
    != shared["label_deepset"]
]

print("Cross-dataset label conflicts:", len(conflicts))

#5. inspection of conficting labels
print("\n===== SAFEGUARD LABEL CONFLICTS =====")

safe = datasets["safeguard"].copy()
deep = datasets["deepset"].copy()

# Find texts with more than one label
conflict_counts = safe.groupby("text")["label"].nunique()

conflicting_texts = conflict_counts[
    conflict_counts > 1
].index

for i, text in enumerate(conflicting_texts, 1):
    print(f"\nConflict {i}:")
    print("Text:", repr(text[:150]))

    matches = safe[safe["text"] == text]

    print(matches[["label", "source_file"]].to_string(index=False))

#6. cross-dataset label conflict check

print("\n===== CROSS-DATASET LABEL COMPARISON =====")

shared = safe[["text", "label"]].drop_duplicates().merge(
    deep[["text", "label"]].drop_duplicates(),
    on="text",
    suffixes=("_safeguard", "_deepset")
)

for i, row in enumerate(shared.itertuples(), 1):
    print(f"\nShared text {i}:")
    print("Text:", repr(row.text[:150]))
    print("Safe-Guard label:", row.label_safeguard)
    print("Deepset label:", row.label_deepset)

    if row.label_safeguard != row.label_deepset:
        print("STATUS: LABEL CONFLICT")
    else:
        print("STATUS: LABELS AGREE")


import re
import unicodedata

# Normalize capitalization and whitespace
def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold()
    text = re.sub(r"\s+", " ", text).strip()
    return text

print("\n===== NORMALIZED DUPLICATE AUDIT =====")

normalized_counts = {}

for name, df in datasets.items():
    normalized = df["text"].map(normalize_text)

    exact_count = df.duplicated(subset=["text"]).sum()
    normalized_count = normalized.duplicated().sum()

    normalized_counts[name] = normalized.value_counts()

    print(f"\nDataset: {name}")
    print("Exact duplicate extra rows:", exact_count)
    print("Normalized duplicate extra rows:", normalized_count)
    print("Additional matches:", normalized_count - exact_count)

print("\n===== CROSS-DATASET NORMALIZED MATCHES =====")

from itertools import combinations

for a, b in combinations(datasets.keys(), 2):
    c1 = normalized_counts[a]
    c2 = normalized_counts[b]

    shared = set(c1.index) & set(c2.index)

    pairs = sum(
        int(c1[t]) * int(c2[t])
        for t in shared
    )

    print(f"{a} vs {b}: {pairs} duplicate pairs")


===== WITHIN-DATASET AUDIT =====

Dataset: safeguard
Total rows: 10296
Columns: ['text', 'label', 'source_file']
Missing text: 0
Class counts:
label
0    7150
1    3146
Name: count, dtype: int64
Duplicate extra rows: 159
Duplicate pairs: 341

Dataset: deepset
Total rows: 662
Columns: ['text', 'label', 'source_file']
Missing text: 0
Class counts:
label
0    399
1    263
Name: count, dtype: int64
Duplicate extra rows: 0
Duplicate pairs: 0

Dataset: gandalf
Total rows: 1000
Columns: ['text', 'similarity', 'source_file']
Missing text: 0
Duplicate extra rows: 0
Duplicate pairs: 0

===== CROSS-DATASET AUDIT =====

safeguard vs deepset
Distinct shared texts: 6
Cross-dataset duplicate pairs: 6
Rows involved in first dataset: 6
Rows involved in second dataset: 6

safeguard vs gandalf
Distinct shared texts: 0
Cross-dataset duplicate pairs: 0
Rows involved in first dataset: 0
Rows involved in second dataset: 0

deepset vs gandalf
Distinct shared texts: 0
Cross-dataset duplicate pairs: 0
Rows inv

thje above code block is our baseline

In [4]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
VALIDATION_FRACTION = 0.20
DATA_DIR = Path.cwd()
PROJECT_DIR = DATA_DIR
RESULTS_DIR = PROJECT_DIR / "results"


def read_downloads():
    """Read every dataset partition, remembering its original row identity."""
    pieces = []
    file_log = []

    for path in sorted(DATA_DIR.rglob("*.parquet")):
        folder = path.parent.name.lower()
        if "safe-guard" in folder:
            dataset = "safeguard"
        elif "deepset" in folder:
            dataset = "deepset"
        elif "gandalf" in folder:
            dataset = "gandalf"
        else:
            continue

        original_partition = path.name.split("-", 1)[0]
        if original_partition not in {"train", "validation", "test"}:
            raise ValueError(f"Unknown publisher partition: {path.name}")

        raw = pd.read_parquet(path)
        if "text" not in raw.columns or raw["text"].isna().any():
            raise ValueError(f"Missing or null text in {path}")

        # Gandalf contains attack prompts and a similarity column, NOT a
        # binary ground-truth label. Treat all as positive for this project.
        if dataset == "gandalf":
            labels = pd.Series(1, index=raw.index, dtype=int)
        else:
            if "label" not in raw.columns:
                raise ValueError(f"Missing label in {path}")
            labels = pd.to_numeric(raw["label"], errors="raise")
            if not labels.isin([0, 1]).all():
                raise ValueError(f"Nonbinary label in {path}")
            labels = labels.astype(int)

        frame = pd.DataFrame({
            "row_id": [f"{dataset}:{path.name}:{i}" for i in range(len(raw))],
            "dataset": dataset,
            "source_partition": original_partition,
            "text": raw["text"].astype(str).to_numpy(),
            "label": labels.to_numpy(),
        })
        pieces.append(frame)
        file_log.append(f"{dataset}\t{original_partition}\t{len(raw)}\t{path.relative_to(DATA_DIR)}")

    if not pieces:
        raise FileNotFoundError(f"No dataset Parquet files found beneath {DATA_DIR}")
    pool = pd.concat(pieces, ignore_index=True)
    if pool["row_id"].duplicated().any():
        raise ValueError("row_id is not unique; inspect the downloaded filenames")

    for name in ("safeguard", "deepset", "gandalf"):
        if name not in set(pool["dataset"]):
            raise FileNotFoundError(f"Missing {name} Parquet files under {DATA_DIR}")

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    (RESULTS_DIR / "files_used.txt").write_text(
        "dataset\toriginal_partition\trows\tfile\n" + "\n".join(file_log) + "\n",
        encoding="utf-8",
    )
    return pool


def main():
    pool = read_downloads()

    # Keep the publisher's Safe-Guard TEST partition unchanged.
    safeguard_test = pool.loc[
        (pool.dataset == "safeguard") & (pool.source_partition == "test")
    ].copy()
    safeguard_train = pool.loc[
        (pool.dataset == "safeguard") & (pool.source_partition == "train")
    ].copy()
    if safeguard_train.empty or safeguard_test.empty:
        raise ValueError("Safe-Guard must have the publisher's train and test files")

    # External datasets are NEVER used for training or threshold tuning.
    # Every publisher partition from Deepset/Gandalf is external test data.
    external_test = pool.loc[pool.dataset.isin(["deepset", "gandalf"])].copy()
    test_df = pd.concat([safeguard_test, external_test], ignore_index=True)
    test_df["split"] = "test"

    # Uniform label policy: leave original Safe-Guard/Deepset labels as given.
    # Remove ALL Safe-Guard training records with internally conflicting labels.
    conflicting = safeguard_train.groupby("text")["label"].nunique()
    conflicting_texts = set(conflicting.index[conflicting > 1])
    without_conflicts = safeguard_train.loc[
        ~safeguard_train.text.isin(conflicting_texts)
    ].copy()
    removed_conflicts = len(safeguard_train) - len(without_conflicts)

    # Protect the fixed tests FIRST (both Safe-Guard and external sets).
    # This also ensures any future validation rows do not overlap tests.
    test_texts = set(test_df["text"])
    without_test_overlap = without_conflicts.loc[
        ~without_conflicts.text.isin(test_texts)
    ].copy()
    removed_test_overlap = len(without_conflicts) - len(without_test_overlap)

    # Exact-match cleaning: retain one row per distinct training-candidate text.
    # These are the only rows that will be divided into train and validation.
    unique_candidates = without_test_overlap.drop_duplicates(
        subset="text", keep="first"
    ).copy()
    removed_train_duplicates = len(without_test_overlap) - len(unique_candidates)

    # Fixed stratified 80/20 split of the distinct Safe-Guard training texts.
    # Because these are unique, identical strings cannot straddle train/val.
    train_df, val_df = train_test_split(
        unique_candidates,
        test_size=VALIDATION_FRACTION,
        random_state=SEED,
        stratify=unique_candidates["label"],
    )
    train_df = train_df.copy()
    val_df = val_df.copy()
    train_df["split"] = "train"
    val_df["split"] = "validation"

    # Verify NO raw-text leakage into any held-out set.
    train_texts = set(train_df.text)
    val_texts = set(val_df.text)
    overlap_train_test = len(train_df.loc[train_df.text.isin(test_texts)])
    overlap_train_val = len(train_df.loc[train_df.text.isin(val_texts)])
    overlap_val_test = len(val_df.loc[val_df.text.isin(test_texts)])
    assert overlap_train_test == 0
    assert overlap_train_val == 0
    assert overlap_val_test == 0

    # One row per kept observation, with the EXACT column names used by the
    # professor's check_checkpoint.py script.
    kept = pd.concat([train_df, val_df, test_df], ignore_index=True)
    if kept.row_id.duplicated().any():
        raise AssertionError("Duplicate row_id in retained observations")
    kept[["row_id", "dataset", "split", "label"]].to_csv(
        PROJECT_DIR / "split.csv", index=False
    )

    # Text-bearing files for baseline.py to train and score. Keep these out
    # of public repositories if any source license restricts redistribution.
    for name, frame in [
        ("train", train_df), ("validation", val_df), ("test", test_df)
    ]:
        frame[["row_id", "dataset", "split", "label", "text"]].to_csv(
            RESULTS_DIR / f"{name}.csv", index=False
        )

    report = kept.groupby(["dataset", "split"], sort=True)["label"].agg(
        rows="size", positive="sum"
    ).reset_index()
    report["negative"] = report["rows"] - report["positive"]
    report["positive_pct"] = (report["positive"] / report["rows"] * 100).round(2)
    report.to_csv(RESULTS_DIR / "split_report.csv", index=False)

    removal_lines = [
        f"Original Safe-Guard publisher train rows: {len(safeguard_train)}",
        f"Dropped: internally conflicting labels: {removed_conflicts} rows ({len(conflicting_texts)} distinct texts)",
        f"Dropped: exact overlap with any fixed test set: {removed_test_overlap} rows",
        f"Dropped: remaining repeated Safe-Guard training texts: {removed_train_duplicates} rows",
        f"Safe-Guard unique rows assigned to train + validation: {len(unique_candidates)}",
        f"Fixed original Safe-Guard test rows: {len(safeguard_test)}",
        f"Fixed external test rows: {len(external_test)}",
        f"Post-cleaning train rows matching any test: {overlap_train_test}",
        f"Post-cleaning train rows matching validation: {overlap_train_val}",
        f"Post-cleaning validation rows matching test: {overlap_val_test}",
        "Note: this exact-match baseline does not yet eliminate character-Jaccard or semantic near-duplicate boundary overlaps.",
        "Note: original test rows, including any within-test duplicates, were kept unchanged.",
    ]
    (RESULTS_DIR / "cleaning_report.txt").write_text(
        "\n".join(removal_lines) + "\n", encoding="utf-8"
    )

    print("\nFINAL EXACT-MATCH BASELINE SPLITS")
    print(report.to_string(index=False))
    print("\nCLEANING AND LEAKAGE REPORT")
    print("\n".join(removal_lines))
    print(f"\nWrote {PROJECT_DIR / 'split.csv'}")
    print(f"Wrote dataset splits and reports to {RESULTS_DIR}")
    print("Next: train baseline.py to produce scores.csv; then run check_checkpoint.py")


if __name__ == "__main__":
    main()



FINAL EXACT-MATCH BASELINE SPLITS
  dataset      split  rows  positive  negative  positive_pct
  deepset       test   662       263       399         39.73
  gandalf       test  1000      1000         0        100.00
safeguard       test  2060       650      1410         31.55
safeguard      train  6464      1938      4526         29.98
safeguard validation  1617       485      1132         29.99

CLEANING AND LEAKAGE REPORT
Original Safe-Guard publisher train rows: 8236
Dropped: internally conflicting labels: 5 rows (2 distinct texts)
Dropped: exact overlap with any fixed test set: 84 rows
Dropped: remaining repeated Safe-Guard training texts: 66 rows
Safe-Guard unique rows assigned to train + validation: 8081
Fixed original Safe-Guard test rows: 2060
Fixed external test rows: 1662
Post-cleaning train rows matching any test: 0
Post-cleaning train rows matching validation: 0
Post-cleaning validation rows matching test: 0
Note: this exact-match baseline does not yet eliminate character

creation of split

In [5]:
from __future__ import annotations

import platform
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import __version__ as sklearn_version
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

SEED = 42
MAX_VALIDATION_FPR = 0.05
REQUIRED_COLUMNS = {"row_id", "dataset", "split", "text", "label"}


def find_project_dir() -> Path:
    """Locate the existing project without __file__ (works in Jupyter)."""
    # User-provided directory comes first; Project is a fallback for the screenshot.
    repo = Path.home() / "source" / "repos"
    candidates = [
        Path(r"C:\Users\Kchau\source\repos\COSC4371\Projects"),
        Path(r"C:\Users\Kchau\source\repos\COSC4371\Project"),
    ]
    for parent in [repo / "COSC4371", repo / "ComputerScience" / "COSC4371",
                   repo / "Computer Science" / "COSC4371"]:
        candidates.extend([parent / "Projects", parent / "Project"])
    candidates.extend([Path.cwd(), *Path.cwd().parents])

    for folder in dict.fromkeys(candidates):
        if (folder / "split.csv").is_file():
            return folder
    raise FileNotFoundError(
        "Cannot find split.csv. Set PROJECT_DIR to the FULL Windows path "
        "of the folder containing split.csv.\n"
        f"Notebook working folder: {Path.cwd()}\n"
        + "Checked:\n" + "\n".join(f"  {p}" for p in dict.fromkeys(candidates))
    )


def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, dtype={"row_id": str, "dataset": str, "split": str})


def load_partitions(project_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, str]:
    """Use existing assignments, never resplit or regenerate split.csv."""
    manifest_path = project_dir / "split.csv"
    manifest = read_csv(manifest_path)

    if REQUIRED_COLUMNS.issubset(manifest.columns):
        rows = manifest.copy()
        source = str(manifest_path)
    else:
        # Some versions of create_splits.py save text in separate CSV files.
        candidate_folders = [project_dir / "results", project_dir]
        candidate_folders += sorted({p.parent for p in project_dir.rglob("train.csv")})
        candidate_folders = list(dict.fromkeys(candidate_folders))
        split_folder = next(
            (folder for folder in candidate_folders
             if all((folder / f"{name}.csv").is_file()
                    for name in ("train", "validation", "test"))),
            None,
        )
        if split_folder is None:
            raise FileNotFoundError(
                "Found split.csv, but it does not include all columns "
                f"{sorted(REQUIRED_COLUMNS)}. It has: {list(manifest.columns)}.\n"
                "Also could not find train.csv, validation.csv, test.csv "
                "together in one folder. Do not train with fabricated rows. "
                "Check the output of create_splits.py."
            )
        frames = [read_csv(split_folder / f"{name}.csv")
                  for name in ("train", "validation", "test")]
        rows = pd.concat(frames, ignore_index=True)
        source = str(split_folder)

    missing = REQUIRED_COLUMNS.difference(rows.columns)
    if missing:
        raise ValueError(f"Input data missing required columns: {sorted(missing)}")

    rows["split"] = rows["split"].str.strip().str.lower().replace(
        {"val": "validation", "valid": "validation"})
    rows["dataset"] = rows["dataset"].str.strip().str.lower().replace(
        {"safe-guard": "safeguard", "safe_guard": "safeguard"})
    rows["label"] = pd.to_numeric(rows["label"], errors="raise")

    if rows["row_id"].isna().any() or rows["row_id"].duplicated().any():
        raise ValueError("row_id must be non-null and unique across all rows.")
    if rows[["text", "label", "dataset", "split"]].isna().any().any():
        raise ValueError("Found missing text, label, dataset, or split values.")
    if not rows["label"].isin([0, 1]).all():
        raise ValueError("Labels must be 0 (benign) or 1 (injection).")
    if not rows["split"].isin(["train", "validation", "test"]).all():
        raise ValueError(f"Unexpected split values: {rows['split'].unique().tolist()}")
    if "row_id" in manifest.columns:
        if len(rows) != len(manifest) or set(rows["row_id"]) != set(manifest["row_id"]):
            raise ValueError("Loaded rows do not match the existing split.csv manifest.")

    train = rows[rows["split"] == "train"].copy()
    val = rows[rows["split"] == "validation"].copy()
    test = rows[rows["split"] == "test"].copy()

    if train.empty or val.empty or test.empty:
        raise ValueError("At least one of train, validation, test is empty.")
    if set(train["dataset"]) != {"safeguard"}:
        raise ValueError("Training must contain only Safe-Guard rows.")
    if set(val["dataset"]) != {"safeguard"}:
        raise ValueError("Validation must contain only Safe-Guard rows.")
    if set(test["dataset"]) != {"safeguard", "deepset", "gandalf"}:
        raise ValueError("Test must contain Safe-Guard, Deepset, and Gandalf.")
    if train["label"].nunique() != 2 or val["label"].nunique() != 2:
        raise ValueError("Safe-Guard training and validation must have both label classes.")
    return train, val, test, source


def threshold_at_fpr(labels: np.ndarray, scores: np.ndarray, target: float) -> float:
    negatives = np.sort(scores[labels == 0])[::-1]
    if len(negatives) == 0:
        raise ValueError("Validation contains no benign rows to calibrate FPR.")
    allowed = int(np.floor(target * len(negatives)))
    return float(np.nextafter(negatives[allowed], np.inf))


def main(project_dir: Path | None = None) -> None:
    project_dir = Path(project_dir) if project_dir is not None else find_project_dir()
    results_dir = project_dir / "results"
    print("Project directory:", project_dir)
    print("Notebook working directory:", Path.cwd())

    train, validation, test, source = load_partitions(project_dir)
    print("Using existing split assignments from:", source)
    print(f"Rows: train={len(train):,}, validation={len(validation):,}, test={len(test):,}")
    print("Test dataset counts:\n", test["dataset"].value_counts().to_string())

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", analyzer="word",
            ngram_range=(1, 2), min_df=2, max_features=50_000,
            sublinear_tf=True,
        )),
        ("classifier", LinearSVC(
            C=1.0, class_weight="balanced", random_state=SEED,
            max_iter=10_000,
        )),
    ])

    start = time.perf_counter()
    model.fit(train["text"].astype(str), train["label"].astype(int))
    fit_seconds = time.perf_counter() - start
    print(f"Model trained in {fit_seconds:.2f} seconds")

    val_y = validation["label"].to_numpy(dtype=int)
    val_scores = model.decision_function(validation["text"].astype(str))
    threshold = threshold_at_fpr(val_y, val_scores, MAX_VALIDATION_FPR)
    val_pred = (val_scores >= threshold).astype(int)
    val_fpr = float(val_pred[val_y == 0].mean())
    val_recall = float(val_pred[val_y == 1].mean())
    print(f"Selected threshold: {threshold!r}")
    print(f"Validation FPR: {val_fpr:.4f} (target <= {MAX_VALIDATION_FPR:.2f})")
    print(f"Validation recall: {val_recall:.4f}")

    # Score the fixed test set once, using the validation threshold.
    test_scores = np.asarray(model.decision_function(test["text"].astype(str)))
    scores_output = test[["row_id", "dataset", "label"]].rename(
        columns={"dataset": "test_set"}).copy()
    scores_output["score"] = test_scores

    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = project_dir / "scores.csv"
    backup_csv = results_dir / "scores_before_baseline.csv"
    if output_csv.exists() and not backup_csv.exists():
        shutil.copy2(output_csv, backup_csv)
        print("Backed up prior scores.csv to:", backup_csv)
    scores_output.to_csv(output_csv, index=False, float_format="%.17g")
    print("Saved:", output_csv)

    test_pred = (test_scores >= threshold).astype(int)
    summary_rows = []
    for dataset in ("safeguard", "deepset", "gandalf"):
        mask = test["dataset"].eq(dataset).to_numpy()
        y = test.loc[mask, "label"].to_numpy(dtype=int)
        pred = test_pred[mask]
        n_negative = int((y == 0).sum())
        summary_rows.append({
            "dataset": dataset,
            "rows": len(y),
            "positive_rows": int((y == 1).sum()),
            "negative_rows": n_negative,
            "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred, zero_division=0),
            "f1": f1_score(y, pred, zero_division=0),
            "fpr": float(pred[y == 0].mean()) if n_negative else np.nan,
        })
    metrics = pd.DataFrame(summary_rows)
    metrics.to_csv(results_dir / "baseline_metrics.csv", index=False)
    print("\nPer-dataset test metrics (FPR=N/A when no benign rows):")
    print(metrics.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

    safe_mask = test["dataset"].eq("safeguard").to_numpy()
    cm = confusion_matrix(test.loc[safe_mask, "label"], test_pred[safe_mask], labels=[0, 1])
    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.imshow(cm, cmap="Blues")
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", fontsize=15,
                    color="white" if cm[r, c] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["Benign (0)", "Injection (1)"])
    ax.set_yticks([0, 1], ["Benign (0)", "Injection (1)"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title("Safe-Guard test confusion matrix")
    fig.tight_layout()
    image_path = results_dir / "safeguard_confusion_matrix.png"
    fig.savefig(image_path, dpi=175)
    plt.close(fig)
    print("Saved:", image_path)

    check_command = (f'python check_checkpoint.py --split split.csv '
                     f'--scores scores.csv --threshold {threshold!r}')
    details = [
        "COSC 4371 checkpoint: TF-IDF + LinearSVC baseline",
        "Positive label: 1 = prompt injection; negative label: 0 = benign",
        f"Source: {source}",
        "Train: Safe-Guard only; validation: Safe-Guard only",
        "Test: Safe-Guard, Deepset, Gandalf; test labels NOT used for threshold selection",
        "TF-IDF: word 1-2 grams, min_df=2, max_features=50000, sublinear_tf=True",
        "LinearSVC: C=1.0, class_weight=balanced, random_state=42, max_iter=10000",
        "Score is the LinearSVC decision_function (not probability)",
        f"Threshold: {threshold!r}",
        f"Validation FPR: {val_fpr:.8f}",
        f"Validation recall: {val_recall:.8f}",
        f"Training time (seconds): {fit_seconds:.4f}",
        f"System: {platform.platform()}",
        f"scikit-learn version: {sklearn_version}",
        "Run from the project folder:", check_command,
    ]
    (results_dir / "model_details.txt").write_text("\n".join(details) + "\n", encoding="utf-8")
    print("Saved:", results_dir / "model_details.txt")
    print("\nCHECKPOINT COMMAND (in a terminal inside your Project folder):")
    print(check_command)


if __name__ == "__main__":
    main()


Project directory: C:\Users\Kchau\source\repos\ComputerScience\COSC4371\Project
Notebook working directory: c:\Users\Kchau\source\repos\ComputerScience\COSC4371\Project
Using existing split assignments from: C:\Users\Kchau\source\repos\ComputerScience\COSC4371\Project\results
Rows: train=6,464, validation=1,617, test=3,722
Test dataset counts:
 dataset
safeguard    2060
gandalf      1000
deepset       662
Model trained in 0.59 seconds
Selected threshold: -0.6409550185601036
Validation FPR: 0.0495 (target <= 0.05)
Validation recall: 1.0000
Backed up prior scores.csv to: C:\Users\Kchau\source\repos\ComputerScience\COSC4371\Project\results\scores_before_baseline.csv
Saved: C:\Users\Kchau\source\repos\ComputerScience\COSC4371\Project\scores.csv

Per-dataset test metrics (FPR=N/A when no benign rows):
  dataset  rows  positive_rows  negative_rows  precision  recall     f1    fpr
safeguard  2060            650           1410     0.9113  0.9954 0.9515 0.0447
  deepset   662            263    